### Standard Coupon

In [1]:
import numpy as np
from numba import njit, prange
from scipy.stats import qmc, norm
import time

# Numba-accelerated core simulation for Fixed Coupon Note with quanto adjustment
@njit(parallel=True, fastmath=True)
def monte_carlo_core(S0_A, S0_B, barrier, coupon_rate, T, r, sigma_A, sigma_B, rho,
                     n_simulations, n_observations, sobol_numbers_A, sobol_numbers_B,
                     q_A, q_B, sigma_FX, rho_A_FX, rho_B_FX):
    dt = T / n_observations
    payoffs_local = np.zeros((n_simulations, n_observations))
    hits_local = np.zeros((n_simulations, n_observations))
    L = np.linalg.cholesky(np.array([[1, rho], [rho, 1]]))  # Cholesky for correlation
    print(L)
    for i in prange(n_simulations):
        S_A = S0_A
        S_B = S0_B
        for j in range(n_observations):
            Z = np.dot(L, np.array([sobol_numbers_A[i, j], sobol_numbers_B[i, j]]))

            # Quanto-adjusted drifts
            quanto_drift_A = r - q_A - rho_A_FX * sigma_A * sigma_FX
            quanto_drift_B = r - q_B - rho_B_FX * sigma_B * sigma_FX

            S_A *= np.exp((quanto_drift_A - 0.5 * sigma_A**2) * dt + sigma_A * np.sqrt(dt) * Z[0])
            S_B *= np.exp((quanto_drift_B - 0.5 * sigma_B**2) * dt + sigma_B * np.sqrt(dt) * Z[1])

            if (S_A >= barrier) and (S_B >= barrier):
                payoffs_local[i, j] = coupon_rate * np.exp(-r * (j + 1))
                hits_local[i, j] = 1

    hits = np.sum(hits_local, axis=0)
    payoffs = np.sum(payoffs_local, axis=0)
    return payoffs, hits

# Wrapper function
def monte_carlo_pricing(S0_A, S0_B, barrier, coupon_rate, T, r, sigma_A, sigma_B, rho,
                        n_simulations, n_observations, Notional, fx_rate, q_A, q_B,
                        sigma_FX, rho_A_FX, rho_B_FX):
    coupon = coupon_rate * fx_rate * Notional
    
    sobol = qmc.Sobol(d=2*n_observations, scramble=True)
    sobol_samples = sobol.random(n=n_simulations)
    normal_samples = norm.ppf(sobol_samples)
    sobol_numbers_A = normal_samples[:, :n_observations]
    sobol_numbers_B = normal_samples[:, n_observations:]
    
    #Alternative
    #sobol_numbers_A = np.random.normal(0, 1, size=(n_simulations, n_observations))
    #sobol_numbers_B = np.random.normal(0, 1, size=(n_simulations, n_observations))

    payoffs, hits = monte_carlo_core(S0_A, S0_B, barrier, coupon, T, r, sigma_A, sigma_B, rho,
                                      n_simulations, n_observations, sobol_numbers_A, sobol_numbers_B,
                                      q_A, q_B, sigma_FX, rho_A_FX, rho_B_FX)

    return np.sum(payoffs)/n_simulations, hits/n_simulations

# Parameters
Notional = 100000
S0_A = 100
S0_B = 100
barrier = 100 * 0.7
coupon_rate = 0.12
T = 5
r = 0.05
sigma_A = 0.15
sigma_B = 0.25
q_A = 0.01
q_B = 0.02
rho = 0.6
n_simulations = 2**20
n_observations = 5
fx_rate = 1.0
# Quanto parameters
sigma_FX = 0.15
rho_A_FX = -0.3
rho_B_FX = -0.4

# Run simulation
start_time = time.time()
price, prob = monte_carlo_pricing(S0_A, S0_B, barrier, coupon_rate, T, r, sigma_A, sigma_B,
                                  rho, n_simulations, n_observations, Notional, fx_rate,
                                  q_A, q_B, sigma_FX, rho_A_FX, rho_B_FX)
end_time = time.time()

print(f"The estimated price of the Fixed Coupon Note is {price:,.2f}")
print(f"Execution time: {end_time - start_time:.2f} seconds.")
print(f"Prob: {prob}")

# Discounted cash flow calculation
dcf = [coupon_rate * Notional * np.exp(-r * i) for i in range(1, n_observations+1)]
result = np.dot(dcf, prob)
print("----------------------------------------------------------------")
print(f"Result: {result:,.2f}")
print(f"DCF * Prob: {np.round(np.multiply(dcf, prob.T),2)}")


[[1.  0. ]
 [0.6 0.8]]
The estimated price of the Fixed Coupon Note is 43,002.28
Execution time: 4.10 seconds.
Prob: [0.92912483 0.85389805 0.80893707 0.77999783 0.7604084 ]
----------------------------------------------------------------
Result: 43,002.28
DCF * Prob: [10605.73  9271.67  8355.1   7663.3   7106.48]


### Quanto Call Option Monte Carlo

In [2]:
import numpy as np
from numba import njit, prange
from scipy.stats import qmc, norm
import time

# Numba-accelerated core simulation with Quanto adjustment and dividend yield
@njit(parallel=True, fastmath=True)
def monte_carlo_core(S0, K, T, r_d, r_f, q, sigma, sigma_fx, rho, FX0, n_simulations, n_observations, sobol_numbers):
    dt = T / n_observations
    payoffs = np.zeros(n_simulations)
    
    # Correct drift term
    drift = r_f - q - rho * sigma * sigma_fx - 0.5 * sigma**2

    for i in prange(n_simulations):
        S = S0
        for j in range(n_observations):
            S *= np.exp(drift * dt + sigma * np.sqrt(dt) * sobol_numbers[i, j])
        payoffs[i] = max(S - K, 0) * FX0  # Convert to domestic currency

    return payoffs

# Wrapper function
def monte_carlo_option_pricing(S0, K, T, r_d, r_f, q, sigma, sigma_fx, rho, FX0, n_simulations, n_observations):
    sobol = qmc.Sobol(d=n_observations, scramble=True)
    sobol_numbers = sobol.random(n=n_simulations)
    sobol_numbers = norm.ppf(sobol_numbers)  # Transform Sobol samples to standard normal

    payoffs = monte_carlo_core(S0, K, T, r_d, r_f, q, sigma, sigma_fx, rho, FX0, n_simulations, n_observations, sobol_numbers)
    return np.mean(payoffs) * np.exp(-r_d * T)



# Parameters
S0 = 95
K = 105
T = 1
r_d = 0.06
r_f = 0.04
q = 0.02
sigma = 0.3
sigma_fx = 0.15
rho = -1
FX0 = 1.1
n_simulations = 2**20
n_observations = 1

# Run simulation
start_time = time.time()
option_price = monte_carlo_option_pricing(S0, K, T, r_d, r_f, q, sigma, sigma_fx, rho, FX0, n_simulations, n_observations)
end_time = time.time()

# Output
print(f"Monte Carlo price of European Quanto call option with dividend yield is {option_price:.2f}")
print(f"Derivativesacademy price of the European Quanto call option with dividend yield is 10.96")
print(f"Execution time: {end_time - start_time:.2f} seconds.")
## https://shiny.derivativesacademy.com/app_direct/quantooptions/

Monte Carlo price of European Quanto call option with dividend yield is 10.96
Derivativesacademy price of the European Quanto call option with dividend yield is 10.96
Execution time: 0.76 seconds.


### Consistency with Analytical Black Scholes

In [3]:
# Black-Scholes analytical formula
def black_scholes_call(S0, K, T, r, q, sigma):
    d1 = (np.log(S0 / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S0 * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

# Check Quanto with European 
# Parameters
S0 = 95
K = 105
T = 1
r_d = 0.06
r_f = 0.06
q = 0.02
sigma = 0.3
sigma_fx = 0.0
rho = 0.0
FX0 = 1.0
n_simulations = 2**20
n_observations = 1

# Run simulation
start_time = time.time()
option_price = monte_carlo_option_pricing(S0, K, T, r_d, r_f, q, sigma, sigma_fx, rho, FX0, n_simulations, n_observations)
end_time = time.time()

# Analytical price
bs_price = black_scholes_call(S0, K, T, r_d, q, sigma)

# Output
print(f"The estimated price of the European Quanto call option with dividend yield is {option_price:.2f}")
print(f"European Call Option Price (Black-Scholes): {bs_price:.2f}")
print(f"Execution time: {end_time - start_time:.2f} seconds.")


The estimated price of the European Quanto call option with dividend yield is 8.79
European Call Option Price (Black-Scholes): 8.79
Execution time: 0.62 seconds.
